# Week 4: AI Agent - LLM이 스스로 도구를 사용하게 하기

지금까지는 우리가 직접 코드를 짜서 "이미지 → 텍스트 → JSON" 순서를 정했습니다.
이번 주에는 **LLM이 스스로 판단하여 도구(함수)를 호출**하는 AI Agent를 만듭니다.

## 핵심 개념
- **Tool Use (Function Calling)**: LLM에게 사용 가능한 도구 목록을 알려주면, LLM이 필요할 때 알아서 호출
- **Agent**: 도구를 사용하여 다단계 작업을 자율적으로 수행하는 LLM 시스템

## 예시
```
사용자: "이 차트에서 진단명을 확인하고, 해당 KCD 코드를 찾아줘"

Agent가 알아서:
  1) OCR 도구로 이미지에서 텍스트 추출
  2) 구조화 도구로 진단명 파싱
  3) KCD 검색 도구로 코드 조회
  4) 결과 종합하여 답변
```

---

## 1. 환경 설정

In [ ]:
!pip install -q openai pillow

import os
import json
import base64
from getpass import getpass
from openai import OpenAI

api_key = getpass("OpenAI API Key를 입력하세요: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

print("설정 완료!")

## 2. 샘플 데이터 준비

In [ ]:
import glob

!git clone --depth 1 https://github.com/{계정}/{레포}.git /content/repo 2>/dev/null || true
!cp /content/repo/emr_samples/*.png /content/ 2>/dev/null || true

samples = sorted(glob.glob("/content/*.png"))
print(f"샘플 {len(samples)}개 준비 완료")

## 3. 도구(Tool) 정의하기

Agent가 사용할 수 있는 도구(함수)들을 정의합니다.

### 도구 목록
1. `ocr_extract` - 이미지에서 텍스트 추출
2. `parse_emr_fields` - 텍스트에서 EMR 필드 구조화
3. `lookup_kcd_code` - 진단명으로 KCD 코드 검색
4. `check_insurance_coverage` - 보험 급여 여부 확인

In [ ]:
# --- 도구 1: OCR 텍스트 추출 ---
def ocr_extract(image_path: str) -> str:
    """EMR 스캔 이미지에서 텍스트를 추출한다."""
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    resp = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "이 의료 문서 이미지에서 모든 텍스트를 정확하게 추출해주세요."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}", "detail": "high"}}
            ]
        }],
        max_tokens=4096,
    )
    return resp.choices[0].message.content


# --- 도구 2: EMR 필드 파싱 ---
def parse_emr_fields(text: str) -> dict:
    """OCR 텍스트에서 환자정보, 진단명, 처방 등을 구조화한다."""
    resp = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{
            "role": "user",
            "content": f"""이 EMR 텍스트에서 주요 필드를 추출하여 JSON으로 구조화하세요.
필드: hospital_name, patient_name, patient_id, visit_date, diagnosis, medications, doctor_name
없는 필드는 null. JSON만 출력.

텍스트:
{text}"""
        }],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


# --- 도구 3: KCD 코드 검색 (시뮬레이션) ---
KCD_DATABASE = {
    "고혈압": {"code": "I10", "name": "본태성(원발성) 고혈압"},
    "본태성 고혈압": {"code": "I10", "name": "본태성(원발성) 고혈압"},
    "당뇨": {"code": "E11", "name": "2형 당뇨병"},
    "제2형 당뇨병": {"code": "E11", "name": "2형 당뇨병"},
    "고지혈증": {"code": "E78.5", "name": "상세불명의 고지혈증"},
    "이상지질혈증": {"code": "E78.5", "name": "상세불명의 고지혈증"},
    "급성 상기도 감염": {"code": "J06.9", "name": "급성 상기도 감염, 상세불명"},
    "감기": {"code": "J06.9", "name": "급성 상기도 감염, 상세불명"},
    "위식도 역류": {"code": "K21", "name": "위식도 역류병"},
    "역류성 식도염": {"code": "K21.0", "name": "식도염을 동반한 위식도 역류병"},
    "요통": {"code": "M54.5", "name": "허리통증"},
    "허리 통증": {"code": "M54.5", "name": "허리통증"},
    "골관절염": {"code": "M17", "name": "무릎관절증"},
    "퇴행성 관절염": {"code": "M17", "name": "무릎관절증"},
}


def lookup_kcd_code(diagnosis_name: str) -> dict:
    """진단명으로 KCD(한국표준질병사인분류) 코드를 검색한다."""
    # 정확 매칭
    if diagnosis_name in KCD_DATABASE:
        return KCD_DATABASE[diagnosis_name]

    # 부분 매칭
    for key, value in KCD_DATABASE.items():
        if key in diagnosis_name or diagnosis_name in key:
            return value

    return {"code": "UNKNOWN", "name": f"'{diagnosis_name}'에 대한 코드를 찾을 수 없습니다"}


# --- 도구 4: 보험 급여 확인 (시뮬레이션) ---
def check_insurance_coverage(kcd_code: str) -> dict:
    """KCD 코드의 건강보험 급여 여부를 확인한다."""
    covered_codes = ["I10", "E11", "E78.5", "J06.9", "K21", "K21.0", "M54.5", "M17"]
    is_covered = kcd_code in covered_codes
    return {
        "kcd_code": kcd_code,
        "is_covered": is_covered,
        "coverage_type": "급여" if is_covered else "비급여",
        "note": "건강보험 적용 대상" if is_covered else "건강보험 미적용 (본인부담)"
    }


print("도구 4개 정의 완료!")
print("  1. ocr_extract - 이미지→텍스트")
print("  2. parse_emr_fields - 텍스트→JSON")
print("  3. lookup_kcd_code - 진단명→KCD코드")
print("  4. check_insurance_coverage - 급여여부 확인")

## 4. OpenAI Tool Use 설정 (핵심!)

위에서 정의한 함수들을 OpenAI API의 `tools` 형식으로 등록합니다.

### Tool Use가 작동하는 원리
1. LLM에게 사용 가능한 도구 목록을 JSON 스키마로 알려줌
2. 사용자 질문을 받으면, LLM이 어떤 도구를 호출할지 판단
3. LLM이 도구 호출 요청을 반환 ("이 함수를 이 인자로 호출해줘")
4. 우리 코드가 실제로 함수를 실행하고 결과를 LLM에 돌려줌
5. LLM이 결과를 바탕으로 최종 답변 생성

In [ ]:
# OpenAI tools 스키마 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "ocr_extract",
            "description": "EMR 스캔 이미지에서 텍스트를 추출한다. 이미지 파일 경로를 받아 텍스트를 반환.",
            "parameters": {
                "type": "object",
                "properties": {
                    "image_path": {"type": "string", "description": "이미지 파일 경로"}
                },
                "required": ["image_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "parse_emr_fields",
            "description": "OCR로 추출한 텍스트에서 환자정보, 진단명, 처방 등을 구조화된 JSON으로 변환한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "OCR 추출 텍스트"}
                },
                "required": ["text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_kcd_code",
            "description": "진단명으로 KCD(한국표준질병사인분류) 코드를 검색한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "diagnosis_name": {"type": "string", "description": "검색할 진단명"}
                },
                "required": ["diagnosis_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_insurance_coverage",
            "description": "KCD 코드의 건강보험 급여 여부를 확인한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "kcd_code": {"type": "string", "description": "KCD 코드 (예: I10)"}
                },
                "required": ["kcd_code"]
            }
        }
    }
]

# 도구 이름 → 실제 함수 매핑
TOOL_MAP = {
    "ocr_extract": ocr_extract,
    "parse_emr_fields": parse_emr_fields,
    "lookup_kcd_code": lookup_kcd_code,
    "check_insurance_coverage": check_insurance_coverage,
}

print("Tool Use 설정 완료!")

## 5. Agent 실행 루프 만들기

LLM이 도구를 호출하면 실행하고, 결과를 다시 LLM에 전달하는 루프입니다.

In [ ]:
def run_agent(user_message, available_files=None):
    """Agent를 실행한다. LLM이 도구를 호출하면 실행하고 결과를 돌려준다."""

    # 시스템 메시지
    system_msg = """당신은 병원 EMR 데이터 처리 전문 AI Agent입니다.
사용자의 요청을 수행하기 위해 주어진 도구들을 활용하세요.
각 단계를 수행할 때 어떤 도구를 왜 사용하는지 설명하세요."""

    if available_files:
        file_list = "\n".join(f"  - {f}" for f in available_files)
        system_msg += f"\n\n사용 가능한 EMR 이미지 파일:\n{file_list}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_message},
    ]

    print(f"\n{'=' * 60}")
    print(f"사용자: {user_message}")
    print(f"{'=' * 60}")

    # Agent 루프 (최대 10번 반복)
    for step in range(10):
        response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=messages,
            tools=tools,
        )

        msg = response.choices[0].message

        # 도구 호출이 없으면 최종 답변
        if not msg.tool_calls:
            print(f"\n--- Agent 최종 답변 ---")
            print(msg.content)
            return msg.content

        # 도구 호출 처리
        messages.append(msg)

        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            print(f"\n[Step {step + 1}] 도구 호출: {fn_name}({fn_args})")

            # 실제 함수 실행
            fn = TOOL_MAP[fn_name]
            result = fn(**fn_args)

            # 결과를 문자열로 변환
            if isinstance(result, dict):
                result_str = json.dumps(result, ensure_ascii=False, indent=2)
            else:
                result_str = str(result)

            print(f"  → 결과: {result_str[:200]}{'...' if len(result_str) > 200 else ''}")

            # 결과를 대화에 추가
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result_str,
            })

    return "Agent가 10단계 내에 완료하지 못했습니다."


print("Agent 함수 정의 완료!")

## 6. Agent 실행 - 기본 예시

Agent에게 자연어로 지시를 내리면, 스스로 도구를 선택하여 작업을 수행합니다.

In [ ]:
# 예시 1: 이미지에서 환자 정보 추출
run_agent(
    "첫 번째 EMR 이미지에서 텍스트를 추출하고, 환자 정보를 정리해줘.",
    available_files=samples
)

In [ ]:
# 예시 2: 진단코드 검색까지 자동 수행
run_agent(
    "첫 번째 EMR 이미지에서 진단명을 확인하고, 해당 KCD 코드와 보험 급여 여부를 알려줘.",
    available_files=samples
)

## 7. (실습) 자유롭게 Agent에게 질문하기

Agent에게 다양한 지시를 내려보세요.

In [ ]:
# === 자유롭게 지시를 수정해보세요 ===
my_instruction = "모든 EMR 이미지를 분석해서, 각 환자의 진단명과 처방약을 비교 정리해줘."
# ====================================

run_agent(my_instruction, available_files=samples)

---

## 정리

### Agent vs 파이프라인 비교
| | Week 3 파이프라인 | Week 4 Agent |
|---|---|---|
| 실행 순서 | 코드로 고정 | LLM이 판단 |
| 유연성 | 정해진 단계만 실행 | 질문에 따라 다른 도구 조합 |
| 적합한 상황 | 대량 일괄 처리 | 다양한 질의에 대응 |
| 비용 | 예측 가능 | 질문에 따라 변동 |

### Tool Use 핵심 구조
```
1. 도구 정의 (함수 + JSON 스키마)
2. LLM에 도구 목록 전달 (tools 파라미터)
3. LLM이 도구 호출 결정 → tool_calls 반환
4. 코드가 실제 함수 실행 → 결과를 LLM에 전달
5. LLM이 결과로 최종 답변 생성
```

### 실무 확장 아이디어
- KCD 코드 DB를 실제 데이터로 교체
- 건강보험 심사평가원 API 연동
- Streamlit으로 웹 UI 구축
- MCP(Model Context Protocol)로 다른 시스템과 연결